# Base vectorial del corpus legal (bge-m3 + FAISS)

Arma los vectores de los dos índices (normas y jurisprudencia), mide con `eval_recuperacion.py` y deja un zip para compartir.
Corre igual en **Colab**, **Kaggle** o un **PC con GPU** (8 GB alcanza). **Guarda el avance**: si se cae, vuelve a correr todo y sigue desde donde quedó.

## Antes de correr

Arma una carpeta `hackathon_vectorial` con:
- lo que trae `kit_vectorial.zip` (los `.py`, `seed_targets.json` y este notebook),
- `corpus_v1.zip` (o `corpus.db` suelto),
- `sample_50.jsonl` (solo para medir; si no está, se salta la medición).

**Colab:** sube la carpeta a *Mi unidad* (`MyDrive/hackathon_vectorial`). Entorno de ejecución → Cambiar tipo → **GPU T4**. Luego *Ejecutar todo*. El avance queda en Drive.

**Kaggle:** crea un *Dataset* con esa carpeta y agrégalo al notebook (*Add Input*). En *Settings*: **Accelerator GPU T4** e **Internet on**.
Lo más seguro es *Save Version → Save & Run All*: corre sola en segundo plano hasta 12 h y guarda lo que produce.
Si se corta, crea otra versión agregando como input la salida de la versión anterior: el notebook recoge los vectores que ya estaban.

**PC con GPU:** pon el notebook dentro de la carpeta y ábrelo con Jupyter o VS Code desde un entorno con `torch` con CUDA.
Si `torch.cuda.is_available()` da `False`, instala torch con CUDA desde pytorch.org antes de seguir.

Solo hay que tocar la celda de configuración, y casi nunca hace falta.

In [ ]:
# ===== Configuracion =====
MODELO = "BAAI/bge-m3"                 # encoder abierto (MIT)
RERANKER = "BAAI/bge-reranker-v2-m3"   # cross-encoder abierto (Apache 2.0)
MAX_LEN = 512                          # tokens por chunk
SHARD = 2000                           # chunks por archivo de avance (no cambiar a mitad de camino)
BATCH = None                           # None = automatico segun la VRAM
CARPETA = None                         # None = automatico (MyDrive/hackathon_vectorial, /kaggle/working/..., o esta carpeta)
MEDIR = True                           # correr eval_recuperacion: BM25, hibrido e hibrido + reranker
INCLUIR_BM25_EN_ZIP = True             # el zip final lleva tambien BM25 (asi nadie tiene que reindexar)
BORRAR_SHARDS_AL_FINAL = True          # los shards ya quedaron dentro de dense.faiss; borrarlos libera ~1,7 GB

In [ ]:
# ===== Entorno y carpeta de trabajo =====
import os
os.environ["JAX_PLATFORMS"] = "cpu"  # bm25s usa JAX si esta instalado (Colab) y JAX se apodera del 75 % de la GPU
import sys, glob, json, shutil, sqlite3, subprocess, time, zipfile
from pathlib import Path

try:
    import google.colab  # noqa
    ENTORNO = "colab"
except ImportError:
    ENTORNO = "kaggle" if os.path.exists("/kaggle/input") else "local"

if ENTORNO == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path(CARPETA or "/content/drive/MyDrive/hackathon_vectorial")
    RAPIDO = Path("/content/rapido")          # disco local: la db se lee mucho mas rapido que desde Drive
    BUSCAR_EN = [WORK, Path("/content")]
elif ENTORNO == "kaggle":
    WORK = Path(CARPETA or "/kaggle/working/hackathon_vectorial")
    RAPIDO = Path("/tmp/rapido")
    BUSCAR_EN = [WORK, Path("/kaggle/input")]
else:
    WORK = Path(CARPETA or Path.cwd())
    RAPIDO = WORK / "data"
    BUSCAR_EN = [WORK]
WORK.mkdir(parents=True, exist_ok=True)
RAPIDO.mkdir(parents=True, exist_ok=True)
INDICES = WORK / "indices"
os.environ["PYTHONIOENCODING"] = "utf-8"
print(f"entorno: {ENTORNO}\ncarpeta de trabajo (aqui queda el avance): {WORK}")

In [ ]:
# ===== Dependencias y GPU =====
import importlib.util

def _falta(mod):  # sin importarlo: importar bm25s aqui cargaria JAX en el kernel
    return importlib.util.find_spec(mod) is None

faltan = [p for m, p in [("bm25s", "bm25s"), ("faiss", "faiss-cpu"), ("sentence_transformers", "sentence-transformers")] if _falta(m)]
if faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

import torch
if not torch.cuda.is_available():
    print("OJO: no hay GPU. En Colab/Kaggle activa la GPU; en un PC instala torch con CUDA. En CPU esto tarda muchisimo.")
    vram = 0
else:
    vram = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)")
if BATCH is None:
    BATCH = 64 if vram >= 14 else 32 if vram >= 7 else 16 if vram >= 3.5 else 8
print(f"batch: {BATCH}")

In [ ]:
# ===== Encontrar el codigo, corpus.db y sample_50 =====
def buscar_archivo(nombre):
    for raiz in BUSCAR_EN:
        if (raiz / nombre).exists():
            return raiz / nombre
        hallados = sorted(glob.glob(str(raiz / "**" / nombre), recursive=True))
        hallados = [h for h in hallados if "/indices/" not in h.replace("\\", "/")]
        if hallados:
            return Path(hallados[0])
    return None

p = buscar_archivo("indexar.py")
if p is None:
    raise SystemExit(f"No encuentro indexar.py en {BUSCAR_EN}. Sube el contenido de kit_vectorial.zip.")
CODIGO = p.parent
for f in ["indexar.py", "buscar.py", "citas.py", "seed_match.py", "eval_recuperacion.py"]:
    assert (CODIGO / f).exists(), f"falta {f} junto a indexar.py ({CODIGO})"
print(f"codigo: {CODIGO}")

def db_sana(p):
    try:
        con = sqlite3.connect(f"file:{p}?mode=ro", uri=True)
        ok = con.execute("PRAGMA quick_check").fetchone()[0] == "ok"
        con.close()
        return ok
    except sqlite3.Error:
        return False

def preparar_db():
    """Deja una corpus.db sana en DB. Se copia a un .tmp y se renombra al final: si se corta, no queda una a medias."""
    tmp = RAPIDO / "corpus.db.tmp"
    suelto = buscar_archivo("corpus.db")
    if suelto is not None and suelto.resolve() != DB.resolve() and db_sana(suelto):
        print(f"copiando {suelto} al disco local...")
        shutil.copy(suelto, tmp)
    else:
        comprimido = buscar_archivo("corpus_v1.zip")
        if comprimido is None:
            raise SystemExit("No encuentro una corpus.db sana ni corpus_v1.zip. Si subiste corpus.db, "
                             "espera a que termine de subir o vuelve a subirla.")
        print(f"descomprimiendo {comprimido}...")
        with zipfile.ZipFile(comprimido) as z:
            miembro = next(n for n in z.namelist() if n.endswith("corpus.db"))
            with z.open(miembro) as src, open(tmp, "wb") as dst:
                shutil.copyfileobj(src, dst, 1 << 24)
    os.replace(tmp, DB)

DB = RAPIDO / "corpus.db"
if ENTORNO == "local" and buscar_archivo("corpus.db") is not None:
    DB = buscar_archivo("corpus.db")              # en el PC se usa donde este
elif not db_sana(DB):
    preparar_db()
if not db_sana(DB):
    raise SystemExit(f"{DB} esta danada. Si viene de corpus_v1.zip, el zip se subio incompleto: vuelve a subirlo.")
print(f"corpus.db: {DB} ({DB.stat().st_size / 2**30:.2f} GB), revisada")

MUESTRA = buscar_archivo("sample_50.jsonl")
print(f"sample_50: {MUESTRA or 'no esta, se salta la medicion'}")

# Kaggle: si como input viene la salida de una version anterior, se recogen sus vectores
if ENTORNO == "kaggle":
    for marca in glob.glob("/kaggle/input/**/indices/*/chunk_ids.json", recursive=True):
        viejo = Path(marca).parent
        nuevo = INDICES / viejo.name
        n_viejo = len(glob.glob(str(viejo / "emb_shards" / "*.npy")))
        n_nuevo = len(glob.glob(str(nuevo / "emb_shards" / "*.npy")))
        if n_viejo > n_nuevo or ((viejo / "dense.faiss").exists() and not (nuevo / "dense.faiss").exists()):
            print(f"recogiendo avance anterior de {viejo.name} ({n_viejo} shards)")
            shutil.copytree(viejo, nuevo, dirs_exist_ok=True)

In [ ]:
# ===== Indexar (retoma sola) =====
INDICES_A_ARMAR = {
    # mismas opciones que reconstruir_desde_seed.py, para que los chunk_ids coincidan con los de BM25
    "index_sin_sentencias": ["--sin-decretos-extra", "--sin-sentencias-extra", "--sin-leyes-ruido"],
    "index_juris": ["--solo-sentencias"],
}
# solo para BM25: el nombre del codigo en la cabecera ("Ley 84 de 1873 (Codigo Civil)") subio el top-3 de 23 a 24/40
SOLO_BM25 = {"index_sin_sentencias": ["--nombre-codigo"]}

def correr(args):
    """Corre un script del codigo mostrando la salida en vivo. Devuelve (codigo, salida)."""
    p = subprocess.Popen([sys.executable, "-u", *map(str, args)], cwd=CODIGO, text=True, encoding="utf-8",
                         errors="replace", stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    salida = []
    try:
        for linea in p.stdout:
            if not linea.strip() or "it/s]" in linea or "s/it]" in linea:
                continue  # barras de progreso
            print(linea, end="")
            salida.append(linea)
        return p.wait(), "".join(salida)
    finally:
        if p.poll() is None:  # si se interrumpe la celda, que el proceso no se quede ocupando la GPU
            p.kill()

def listo(out):
    info = out / "info.json"
    return (out / "dense.faiss").exists() and info.exists() and json.loads(info.read_text())["modelo"] == MODELO

for nombre, opciones in INDICES_A_ARMAR.items():
    out = INDICES / nombre
    base = ["indexar.py", "--db", DB, "--out", out, "--modelo", MODELO, *opciones]
    print(f"\n===== {nombre} =====")
    if listo(out):
        print("ya estaba listo")
        continue
    if not (out / "bm25").exists():
        cod, _ = correr(base + ["--solo-bm25", *SOLO_BM25.get(nombre, [])])
        assert cod == 0, "fallo BM25"
    b = BATCH
    while True:
        t = time.time()
        cod, salida = correr(base + ["--solo-denso", "--batch", b, "--shard", SHARD, "--max-len", MAX_LEN])
        if cod == 0:
            print(f"listo en {(time.time() - t) / 60:.0f} min")
            break
        if "out of memory" in salida.lower() and b > 1:
            b //= 2
            print(f"\nsin memoria en la GPU: sigo con batch {b} (lo ya hecho se conserva)")
            torch.cuda.empty_cache()
            continue
        raise SystemExit(f"fallo indexar {nombre}; vuelve a correr esta celda y retoma desde el ultimo shard")

In [ ]:
# ===== Medir con sample_50 (solo para medir: nunca se indexa) =====
# Referencia v1: BM25 solo = 26/40 en top-10 y 19/40 en top-3.
if MEDIR and MUESTRA is not None:
    comun = ["eval_recuperacion.py", "--muestra", MUESTRA, "--db", DB,
             "--index", INDICES / "index_sin_sentencias", "--index-juris", INDICES / "index_juris"]
    corridas = {"BM25": ["--solo-bm25"], "hibrido": [], "hibrido + reranker": ["--reranker", RERANKER]}
    if torch.cuda.is_available():
        libre = torch.cuda.mem_get_info()[0] / 2**30
        print(f"GPU libre: {libre:.1f} GB")
        if libre < 4:
            print("OJO: otro proceso esta ocupando la GPU. Reinicia la sesion y vuelve a ejecutar todo "
                  "(los indices ya hechos no se rehacen). Si no, se mide en CPU, mas lento.")
    informe = [f"# Eval {time.strftime('%Y-%m-%d %H:%M')} | {MODELO} | {ENTORNO}"]
    for nombre, extra in corridas.items():
        print(f"\n===== {nombre} =====")
        t = time.time()
        cod, salida = correr(comun + extra)
        if cod != 0 and "out of memory" in salida.lower():
            print("\nsin memoria en la GPU: repito en CPU")
            nombre += " (CPU)"
            cod, salida = correr(comun + extra + ["--device", "cpu"])
        lineas = [l for l in salida.splitlines() if "top-" in l or "referencias" in l]
        informe.append(f"{nombre} ({time.time() - t:.0f} s): " + " | ".join(l.strip() for l in lineas) if cod == 0 else f"{nombre}: FALLO")
    with open(WORK / "resultados_eval.txt", "a", encoding="utf-8") as f:
        f.write("\n".join(informe) + "\n\n")
    print("\n" + "\n".join(informe) + f"\n\n(guardado en {WORK / 'resultados_eval.txt'})")
else:
    print("medicion saltada")

In [ ]:
# ===== Zip para compartir =====
# Cada companero lo descomprime dentro de data/ y queda data/index_sin_sentencias y data/index_juris con dense.faiss.
nombre_zip = WORK / f"vectores_{MODELO.split('/')[-1]}.zip"
with zipfile.ZipFile(nombre_zip, "w", zipfile.ZIP_STORED) as z:  # los vectores casi no se comprimen
    for nombre in INDICES_A_ARMAR:
        out = INDICES / nombre
        for f in ["dense.faiss", "info.json", "chunk_ids.json"]:
            z.write(out / f, f"{nombre}/{f}")
        if INCLUIR_BM25_EN_ZIP:
            for f in (out / "bm25").rglob("*"):
                z.write(f, f"{nombre}/bm25/{f.relative_to(out / 'bm25').as_posix()}")
    guia = buscar_archivo("GUIA_VECTORES.md")
    if guia is not None:
        z.write(guia, "GUIA_VECTORES.md")
print(f"{nombre_zip} ({nombre_zip.stat().st_size / 2**30:.2f} GB)")

if BORRAR_SHARDS_AL_FINAL:
    for nombre in INDICES_A_ARMAR:
        if listo(INDICES / nombre):
            shutil.rmtree(INDICES / nombre / "emb_shards", ignore_errors=True)
    print("shards borrados")

## Usarlo después

```powershell
# en el repo, con el zip descomprimido en data\
python buscar.py "requisitos de la accion de tutela" --index data/index_sin_sentencias --index-juris data/index_juris
python buscar.py "..." --index data/index_sin_sentencias --index-juris data/index_juris --reranker
```

`buscar.py` usa la GPU sola si hay; en CPU el encoder de la consulta anda bien, el reranker no.